<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Theoretical Foundations

This notebook formalizes the mathematical and algorithmic basis of the corresponding laboratory implementation. The section order mirrors the experimental workflow so that assumptions, estimation steps, diagnostics, and validation criteria remain directly traceable to the executable notebook.

### Technical Context

A camera converts a 3D point into a 2D pixel. Calibration works in the opposite direction: from many known 3D/planar points and their measured image positions, we estimate the camera parameters that produced those pixels.

The implemented pipeline is

\`\`\`text
Calibration images
        ↓
Chessboard corners
        ↓
Known planar coordinates
        ↓
Point normalization
        ↓
Homography H for each image
        ↓
Zhang constraints from all H
        ↓
Intrinsic matrix K
        ↓
Pose R, t for each image
        ↓
Reprojection
        ↓
Error analysis + validation
\`\`\`

### Core Camera Model

For a world point

$$
\mathbf{X}_w=[X,Y,Z,1]^T
$$

and its image observation

$$
\mathbf{x}=[u,v,1]^T,
$$

the pinhole camera model is

$$
s\mathbf{x}
=
K
\begin{bmatrix}
R&t
\end{bmatrix}
\mathbf{X}_w.
$$

This single equation contains almost the entire project.

- $K$ describes the **camera itself**.
- $R$ and $t$ describe **where the camera is relative to the chessboard**.
- $s$ is the unknown homogeneous scale.
- $(u,v)$ are pixel coordinates.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $\mathbf{X}_w$ | homogeneous world point |
| $\mathbf{X}_p=[X,Y,1]^T$ | homogeneous point on the planar chessboard |
| $\mathbf{x}=[u,v,1]^T$ | homogeneous image point |
| $K$ | intrinsic camera matrix |
| $R$ | rotation matrix |
| $t$ | translation vector |
| $H$ | plane-to-image homography |
| $H_n$ | homography estimated in normalized coordinates |
| $Q$ | DLT linear system matrix |
| $\mathbf{h}$ | vectorized homography unknown |
| $V$ | Zhang constraint matrix |
| $b$ | vector containing the independent coefficients of $B$ |
| $B=K^{-T}K^{-1}$ | symmetric matrix used by Zhang's method |
| $\alpha,\beta$ | focal scale factors in pixels |
| $\gamma$ | skew |
| $(u_0,v_0)$ | principal point |
| $\lambda$ | scale used in intrinsic recovery |
| $\lambda_p$ | scale used in pose recovery |
| $e_i$ | reprojection error of point $i$ |

### Analytical Scope

By the end, you should be able to explain without memorization:

- why a planar chessboard creates a homography;
- why point normalization is necessary;
- how DLT turns geometry into $Q\mathbf{h}=0$;
- why SVD solves that homogeneous system;
- how several homographies reveal one common $K$;
- why Zhang uses orthogonality of rotation columns;
- how $R$ and $t$ are recovered from $K^{-1}H$;
- what reprojection error tells us;
- and what can still be wrong even when the code runs.

## 1. Load the Sorted JPEG Calibration Images

### Objective

This task does not estimate any camera parameter yet. It establishes the **experimental dataset** used by all later mathematics.

Each image is one observation of the same chessboard from a different camera pose. Every valid image will eventually contribute one homography $H_i$.

If there are $m$ valid images, then conceptually we obtain

$$
H_1,H_2,\ldots,H_m.
$$

These multiple homographies are essential because the intrinsic matrix $K$ is shared by every image, while $R_i$ and $t_i$ change from view to view.

### Multi-view Requirement

One planar image gives only limited information about the camera. By tilting and translating the chessboard relative to the camera, we create geometrically different constraints.

A good calibration dataset contains views with:

- different orientations;
- different positions;
- visible perspective deformation;
- the entire internal-corner grid detectable.

If every image is almost identical, the equations become poorly conditioned even if many images are available.

### Reproducibility and Ordering

Sorting does **not** change the calibration mathematics. It gives reproducibility:

$$
\text{same files}+\text{same order}
\Rightarrow
\text{same sequence of outputs}.
$$

That matters for debugging, comparing results, and matching figures to image names.

### Project-specific values

The implementation expects:

- JPEG images;
- a repository-relative input directory;
- at least **3 valid views** after corner detection.


## 2. Detect and Refine Chessboard Corners

### Calibration Feature Definition

A calibration chessboard contains alternating black and white squares. The useful points are the **internal intersections** where four squares meet.

For an $8\times6$ internal-corner pattern, each valid image provides

$$
N=8\times6=48
$$

measured image points.

For corner $i$,

$$
\mathbf{x}_i=
\begin{bmatrix}
u_i\\
v_i
\end{bmatrix}
$$

contains its pixel coordinates.

### Feature Suitability

Calibration needs pairs of corresponding points:

$$
\text{known point on chessboard}
\longleftrightarrow
\text{measured pixel}.
$$

Chessboard corners are ideal because they are:

- easy to identify;
- arranged in a known grid;
- repeatable across images;
- geometrically precise.

### Pixel accuracy vs sub-pixel accuracy

Initial corner detection gives a position close to the corner, but camera calibration is sensitive to small coordinate errors.

Suppose the true corner is

$$
(542.37,\;311.82)
$$

pixels.

A pixel-only detector might return

$$
(542,\;312),
$$

while sub-pixel refinement estimates a fractional location much closer to the true geometric intersection.

The implementation therefore uses:

1. \`cv2.findChessboardCorners\`
2. \`cv2.cornerSubPix\`

### Error Propagation

The detected corner coordinates affect:

$$
Q
\rightarrow
H
\rightarrow
V
\rightarrow
K
\rightarrow
R,t
\rightarrow
\text{reprojection error}.
$$

So a small systematic corner error can propagate through the full pipeline.

### Correspondence Ordering

The order of the detected image corners must match the order of the generated planar coordinates.

If point 17 in the image is accidentally paired with point 18 in the world grid, the homography equations are wrong even though both coordinates individually look valid.

### Failure Mode

**Mistake:** thinking the chessboard has $8\times6$ squares.  
Here, $8\times6$ refers to **internal corners**, so the physical board contains one more square along each direction.


## 3. Build the Planar World Coordinates

### From pixels to known geometry

The image gives us $(u,v)$. We also need the corresponding coordinates on the physical chessboard.

Because the chessboard is planar,

$$
Z=0.
$$

So each calibration point can be represented by only two physical coordinates:

$$
(X,Y).
$$

The implementation uses square size

$$
s_q=0.03\ \mathrm{m}.
$$

### Coordinate convention

The first internal corner is chosen as the origin:

$$
(X,Y)=(0,0).
$$

The next corner along the 8-corner direction is

$$
(0.03,0),
$$

then

$$
(0.06,0),
$$

and so on.

Moving one row in the 6-corner direction gives

$$
(0,0.03).
$$

### Homogeneous planar coordinates

For projective geometry, we write

$$
\mathbf{X}_p=
\begin{bmatrix}
X\\
Y\\
1
\end{bmatrix}.
$$

The final component $1$ allows translations and perspective mappings to be expressed with matrix multiplication.

### Why the absolute origin does not matter

We could choose a different corner as $(0,0)$. Calibration would still work if the convention is consistent.

What matters is:

- correct relative geometry;
- correct square size;
- correct correspondence order.

### Why square size matters

The intrinsic matrix $K$ is mainly expressed in pixels, so its focal scales do not depend strongly on the chosen physical unit.

But translation $t$ does.

If square size is specified in metres, then the recovered translation is in metres.


## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

### Numerical Conditioning

DLT solves a linear system using coordinates that may have very different magnitudes.

Typical image coordinates might be

$$
u\approx1000,\qquad v\approx700,
$$

while planar coordinates might be

$$
X\approx0.15,\qquad Y\approx0.09.
$$

Putting numbers with very different scales into the same linear system can produce poor numerical conditioning.

### Hartley-style normalization

For a set of 2D points, compute the centroid

$$
(\bar x,\bar y).
$$

Subtract it:

$$
x_i' = x_i-\bar x,
\qquad
y_i' = y_i-\bar y.
$$

Then compute the mean distance to the origin:

$$
\bar d
=
\frac{1}{N}
\sum_{i=1}^{N}
\sqrt{(x_i')^2+(y_i')^2}.
$$

Choose the scale

$$
s_n=\frac{\sqrt{2}}{\bar d}.
$$

The normalization matrix is

$$
T=
\begin{bmatrix}
s_n&0&-s_n\bar x\\
0&s_n&-s_n\bar y\\
0&0&1
\end{bmatrix}.
$$

### Conditioning Effect

After transformation,

$$
\tilde{\mathbf{x}}_i=T\mathbf{x}_i,
$$

the points have approximately:

- centroid at $(0,0)$;
- mean distance $\sqrt{2}$.

Why $\sqrt{2}$?

Because in 2D it gives coordinates of order one, which is numerically convenient.

### Two separate transforms

The implementation computes:

$$
T_{\mathrm{image}}
$$

for image points, and

$$
T_{\mathrm{plane}}
$$

for chessboard points.

They are different because the two coordinate systems have different units and scales.

### Numerical Illustration

Suppose the centroid is $(500,300)$ and the mean distance is $200$ pixels.

Then

$$
s_n=\frac{\sqrt2}{200}\approx0.00707.
$$

A point near $(600,300)$ becomes approximately

$$
(0.707,0),
$$

which is far better scaled for numerical linear algebra than $(600,300)$.

### Failure Mode

**Mistake:** normalizing image points but not planar points.

Both sides of the homography estimation should be normalized for best conditioning.


## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

### Homography Formulation

Because the calibration target is planar, a point on the chessboard and its image are related by a $3\times3$ homography:

$$
s
\begin{bmatrix}
u\\
v\\
1
\end{bmatrix}
=
H
\begin{bmatrix}
X\\
Y\\
1
\end{bmatrix}.
$$

Write

$$
H=
\begin{bmatrix}
h_{11}&h_{12}&h_{13}\\
h_{21}&h_{22}&h_{23}\\
h_{31}&h_{32}&h_{33}
\end{bmatrix}.
$$

Then

$$
u=
\frac{h_{11}X+h_{12}Y+h_{13}}
     {h_{31}X+h_{32}Y+h_{33}},
$$

$$
v=
\frac{h_{21}X+h_{22}Y+h_{23}}
     {h_{31}X+h_{32}Y+h_{33}}.
$$

### Remove the denominator

Multiply both equations by the denominator. Each correspondence gives two linear equations in the nine unknown homography coefficients.

For normalized $(X,Y)\leftrightarrow(u,v)$, the implementation uses

$$
\begin{bmatrix}
X&Y&1&0&0&0&-uX&-uY&-u
\end{bmatrix},
$$

and

$$
\begin{bmatrix}
0&0&0&X&Y&1&-vX&-vY&-v
\end{bmatrix}.
$$

Stacking all $N$ correspondences gives

$$
Q\mathbf{h}=0,
$$

where

$$
\mathbf{h}
=
[h_{11},h_{12},h_{13},h_{21},h_{22},h_{23},h_{31},h_{32},h_{33}]^T.
$$

With 48 corners,

$$
Q\in\mathbb{R}^{96\times9}.
$$

### Homogeneous Formulation

The homography is defined only up to scale:

$$
H
\quad\text{and}\quad
cH
$$

represent the same projective mapping for any non-zero $c$.

Therefore the trivial scale cannot be determined from the equations.

### SVD Solution

Compute

$$
Q=U\Sigma V^T.
$$

The right singular vector associated with the smallest singular value points in the direction that minimizes

$$
\lVert Q\mathbf{h}\rVert
$$

subject to a normalization such as

$$
\lVert\mathbf{h}\rVert=1.
$$

So

$$
\mathbf{h}=V_{:,-1}.
$$

The implementation reshapes this vector into

$$
H_n.
$$

### Minimum vs practical number of points

A homography has 8 degrees of freedom because scale is arbitrary.

Four point correspondences provide eight independent scalar equations in the ideal case.

But the implementation uses 48 points, making the solution much more robust to noise.

### Failure Modes

- forgetting homogeneous scale ambiguity;
- pairing points in the wrong order;
- using nearly collinear points;
- skipping normalization;
- taking the wrong singular vector.


## 6. Denormalize Each Homography

### Intermediate Estimate

Task 5 estimates a homography in the normalized coordinate systems:

$$
\tilde{\mathbf{x}}
=
H_n
\tilde{\mathbf{X}}_p.
$$

But

$$
\tilde{\mathbf{x}}
=
T_{\mathrm{image}}\mathbf{x},
$$

and

$$
\tilde{\mathbf{X}}_p
=
T_{\mathrm{plane}}\mathbf{X}_p.
$$

Substitute:

$$
T_{\mathrm{image}}\mathbf{x}
=
H_n
T_{\mathrm{plane}}\mathbf{X}_p.
$$

Multiply by $T_{\mathrm{image}}^{-1}$:

$$
\mathbf{x}
=
T_{\mathrm{image}}^{-1}
H_n
T_{\mathrm{plane}}
\mathbf{X}_p.
$$

Therefore

$$
H
=
T_{\mathrm{image}}^{-1}
H_n
T_{\mathrm{plane}}.
$$

This is exactly the formula used in the implementation.

### Fix the arbitrary scale

Since $H$ is only defined up to non-zero scale, the code sets

$$
H_{33}=1
$$

by computing

$$
H
\leftarrow
\frac{H}{H_{33}}.
$$

### Degeneracy Check

If

$$
H_{33}\approx0,
$$

division would be unstable or undefined.

The code therefore treats this as a degenerate case.


## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

### Zhang Constraint Formulation

So far we have estimated a separate homography for each image.

Now we ask:

**What do all these homographies have in common?**

They were produced by the **same camera**, so they share the same intrinsic matrix $K$.

For a planar target,

$$
H
=
K
\begin{bmatrix}
\mathbf{r}_1&
\mathbf{r}_2&
t
\end{bmatrix}.
$$

Let

$$
H=
[\mathbf{h}_1\;\mathbf{h}_2\;\mathbf{h}_3].
$$

Then, up to one scale factor,

$$
K^{-1}\mathbf{h}_1
\propto
\mathbf{r}_1,
$$

$$
K^{-1}\mathbf{h}_2
\propto
\mathbf{r}_2.
$$

### The rotation properties Zhang exploits

Columns of a true rotation matrix are orthonormal:

$$
\mathbf{r}_1^T\mathbf{r}_2=0,
$$

and

$$
\lVert\mathbf{r}_1\rVert
=
\lVert\mathbf{r}_2\rVert.
$$

Substituting the homography relationships gives

$$
\mathbf{h}_1^T
K^{-T}K^{-1}
\mathbf{h}_2
=
0,
$$

and

$$
\mathbf{h}_1^T
K^{-T}K^{-1}
\mathbf{h}_1
-
\mathbf{h}_2^T
K^{-T}K^{-1}
\mathbf{h}_2
=
0.
$$

Define

$$
B=K^{-T}K^{-1}.
$$

Because $B$ is symmetric,

$$
B=
\begin{bmatrix}
B_{11}&B_{12}&B_{13}\\
B_{12}&B_{22}&B_{23}\\
B_{13}&B_{23}&B_{33}
\end{bmatrix},
$$

so it has only six independent coefficients.

Collect them into

$$
b=
[B_{11},B_{12},B_{22},B_{13},B_{23},B_{33}]^T.
$$

### Turn each geometric constraint into a linear equation

For homography columns $\mathbf{h}_i$ and $\mathbf{h}_j$, define

$$
v_{ij}=
\begin{bmatrix}
h_{i1}h_{j1}\\
h_{i1}h_{j2}+h_{i2}h_{j1}\\
h_{i2}h_{j2}\\
h_{i3}h_{j1}+h_{i1}h_{j3}\\
h_{i3}h_{j2}+h_{i2}h_{j3}\\
h_{i3}h_{j3}
\end{bmatrix}.
$$

Then

$$
\mathbf{h}_i^TB\mathbf{h}_j
=
v_{ij}^Tb.
$$

So each image gives two equations:

$$
v_{12}^Tb=0,
$$

$$
(v_{11}-v_{22})^Tb=0.
$$

Stack all images:

$$
Vb=0.
$$

### How many views are theoretically needed?

There are six entries in $b$, but homogeneous scale means only five independent degrees of freedom.

Each image contributes two equations.

So at least three suitably different views are needed:

$$
3\times2=6
$$

constraints.

That is why the implementation requires at least 3 valid views.

### Solve again with SVD

Exactly as with DLT:

$$
V=U\Sigma V^T,
$$

and the last right singular vector gives the homogeneous solution $b$.

### Geometric Interpretation

DLT used many corner correspondences to estimate **one homography**.

Zhang now uses many homographies to estimate **one camera**.

That hierarchy is:

$$
\text{corners}
\rightarrow
H_i
\rightarrow
K.
$$


## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

### The intrinsic matrix

The implementation uses

$$
K=
\begin{bmatrix}
\alpha&\gamma&u_0\\
0&\beta&v_0\\
0&0&1
\end{bmatrix}.
$$

Interpretation:

- $\alpha$ — horizontal focal scale in pixels;
- $\beta$ — vertical focal scale in pixels;
- $\gamma$ — skew between image axes;
- $u_0$ — horizontal principal point;
- $v_0$ — vertical principal point.

For modern cameras, $\gamma$ is usually close to zero, but the implementation estimates it rather than forcing it to zero.

### Recover parameters from $b$

Write

$$
b=
[b_{11},b_{12},b_{22},b_{13},b_{23},b_{33}]^T.
$$

Define

$$
d=b_{11}b_{22}-b_{12}^2.
$$

Then the vertical principal point is

$$
v_0
=
\frac{b_{12}b_{13}-b_{11}b_{23}}{d}.
$$

The intrinsic recovery scale is

$$
\lambda
=
b_{33}
-
\frac{
b_{13}^2
+
v_0(b_{12}b_{13}-b_{11}b_{23})
}{
b_{11}
}.
$$

Now

$$
\alpha
=
\sqrt{
\frac{\lambda}{b_{11}}
},
$$

$$
\beta
=
\sqrt{
\frac{\lambda b_{11}}{d}
},
$$

$$
\gamma
=
-\frac{
b_{12}\alpha^2\beta
}{
\lambda
},
$$

and

$$
u_0
=
\frac{\gamma v_0}{\beta}
-
\frac{
b_{13}\alpha^2
}{
\lambda
}.
$$

### Homogeneous Sign Ambiguity

The equation

$$
Vb=0
$$

is homogeneous.

If $b$ is a solution, then

$$
-b
$$

is also a solution.

SVD may therefore return either sign.

But square-root expressions such as

$$
\frac{\lambda}{b_{11}}
$$

must be positive for a physical real-valued calibration.

The implementation detects an invalid sign configuration and replaces

$$
b\leftarrow-b.
$$

This does not change the homogeneous solution—it only chooses the physically usable orientation of that solution vector.

### Sanity checks on $K$

A reasonable $K$ should satisfy:

- $\alpha>0$;
- $\beta>0$;
- finite entries;
- bottom row approximately $[0,0,1]$;
- principal point often near the image centre;
- skew usually small.

These are plausibility checks, not absolute laws.


## 9. Recover $R$ and $t$ for Every Retained View

### Intrinsics vs extrinsics

Now we know $K$.

For each image,

$$
H
=
K
[\mathbf{r}_1\;\mathbf{r}_2\;t].
$$

Multiply by $K^{-1}$:

$$
K^{-1}H
=
[\tilde{\mathbf{r}}_1\;\tilde{\mathbf{r}}_2\;\tilde t].
$$

These columns are correct only up to a common scale.

### Recover the scale

The implementation uses the first homography column:

$$
\lambda_p
=
\frac{1}{
\lVert
K^{-1}\mathbf{h}_1
\rVert
}.
$$

Then

$$
\mathbf{r}_1
=
\lambda_p
K^{-1}\mathbf{h}_1,
$$

$$
\mathbf{r}_2
=
\lambda_p
K^{-1}\mathbf{h}_2,
$$

$$
t
=
\lambda_p
K^{-1}\mathbf{h}_3.
$$

The third rotation column follows from the right-handed coordinate system:

$$
\mathbf{r}_3
=
\mathbf{r}_1
\times
\mathbf{r}_2.
$$

### Rotation Correction

Because $H$ was estimated from noisy measurements, the matrix

$$
R_{\mathrm{approx}}
=
[\mathbf{r}_1\;\mathbf{r}_2\;\mathbf{r}_3]
$$

will not satisfy exactly

$$
R^TR=I.
$$

### Project to the nearest rotation matrix

Compute

$$
R_{\mathrm{approx}}
=
U\Sigma V^T.
$$

Then

$$
R=UV^T.
$$

This is the orthogonal matrix closest to $R_{\mathrm{approx}}$ in the least-squares/Frobenius sense.

### Enforce a proper rotation

A physical 3D rotation must satisfy

$$
\det(R)=+1.
$$

If

$$
\det(R)=-1,
$$

we have a reflection.

The implementation flips the sign of the last column of $U$ and recomputes $R$.

### Translation Interpretation

With

$$
\mathbf{X}_c
=
R\mathbf{X}_w+t,
$$

$t$ is the chessboard-origin position expressed in camera coordinates.

Because world coordinates are measured in metres, $t$ is also in metres.

### Camera centre

The camera centre in world coordinates is

$$
C=-R^Tt.
$$

This is what the pose visualization plots.


## 10. Reproject the $Z=0$ Calibration Points

### Reprojection Validation

Calibration is not complete just because we computed $K$, $R$ and $t$.

We must ask:

**If these parameters are correct, where do they predict each known chessboard corner should appear?**

That prediction is reprojection.

### Step 1 — Embed the planar point in 3D

A chessboard point is

$$
\mathbf{X}_{3D}
=
\begin{bmatrix}
X\\
Y\\
0
\end{bmatrix}.
$$

### Step 2 — Transform world to camera coordinates

$$
\mathbf{X}_c
=
R\mathbf{X}_{3D}+t.
$$

If

$$
\mathbf{X}_c=
\begin{bmatrix}
X_c\\
Y_c\\
Z_c
\end{bmatrix},
$$

then $Z_c$ represents depth from the camera.

### Step 3 — Apply intrinsics

$$
\tilde{\mathbf{x}}
=
K\mathbf{X}_c.
$$

Write

$$
\tilde{\mathbf{x}}
=
\begin{bmatrix}
\tilde u\\
\tilde v\\
\tilde w
\end{bmatrix}.
$$

### Step 4 — Dehomogenize

Pixel coordinates are

$$
\hat u
=
\frac{\tilde u}{\tilde w},
$$

$$
\hat v
=
\frac{\tilde v}{\tilde w}.
$$

So the predicted pixel is

$$
\hat{\mathbf{x}}
=
[\hat u,\hat v]^T.
$$

### Model Consistency

Reprojection uses the entire estimated model:

$$
K,\ R,\ t.
$$

A poor homography, poor intrinsics, or poor pose will eventually appear as displaced reprojected points.

### Implementation equivalence

The code uses row-vector NumPy operations:

$$
\texttt{world\_xyz @ R.T + t}
$$

which is numerically equivalent to the column-vector equation

$$
R\mathbf{X}+t.
$$


## 11. Compute Point-wise Errors, Mean Error and RMSE

### Point-wise reprojection error

For measured pixel position

$$
\mathbf{x}_i
=
[u_i,v_i]^T
$$

and predicted pixel position

$$
\hat{\mathbf{x}}_i
=
[\hat u_i,\hat v_i]^T,
$$

the residual vector is

$$
\mathbf{r}_i
=
\hat{\mathbf{x}}_i-\mathbf{x}_i.
$$

The implementation uses its Euclidean magnitude:

$$
e_i
=
\lVert
\mathbf{r}_i
\rVert_2
=
\sqrt{
(\hat u_i-u_i)^2
+
(\hat v_i-v_i)^2
}.
$$

The unit is **pixels**.

### Mean reprojection error

For $N$ points,

$$
\bar e
=
\frac1N
\sum_{i=1}^{N}e_i.
$$

This answers:

**How far away is a typical predicted corner?**

### RMSE

$$
\mathrm{RMSE}
=
\sqrt{
\frac1N
\sum_{i=1}^{N}e_i^2
}.
$$

RMSE penalizes large errors more strongly than the arithmetic mean.

### Example

Suppose four errors are

$$
[0.2,\ 0.3,\ 0.4,\ 2.0]\ \mathrm{px}.
$$

The last error is an outlier.

The mean increases, but RMSE increases even more because the $2.0$ value is squared.

That makes RMSE useful for detecting occasional poor fits.

### Per-view vs global metrics

The notebook computes:

- mean error for each image;
- RMSE for each image;
- global mean after concatenating all corner errors;
- global RMSE after concatenating all corner errors.

This lets us answer two different questions:

1. **Is the whole calibration good?**
2. **Is one particular view much worse than the others?**

### What is a “good” reprojection error?

There is no universal threshold.

Interpretation depends on:

- image resolution;
- lens distortion;
- corner quality;
- target quality;
- calibration model.

A smaller value is better, but the most important concept is consistency and absence of systematic residual patterns.

### Failure Mode

A low global average can hide one bad view.

Always inspect both:

$$
\text{global metrics}
\quad\text{and}\quad
\text{per-view metrics}.
$$

## 12. Produce and Save the Six Required Diagnostic Figures

Numbers alone are not enough. Visual diagnostics tell us **why** a calibration is good or bad.

### 12.1 Homography Estimation Pipeline

This diagram summarizes

$$
\text{points}
\rightarrow
T
\rightarrow
Q
\rightarrow
\mathrm{SVD}
\rightarrow
H_n
\rightarrow
H.
$$

Its purpose is conceptual traceability.

### 12.2 Mean Reprojection Error by View

This plot compares

$$
\bar e_1,\bar e_2,\ldots,\bar e_m.
$$

A single bar much larger than the others may indicate:

- weak corner detection;
- motion blur;
- extreme perspective;
- poor geometric conditioning.

### 12.3 Reprojection Error Distribution

The histogram shows the distribution of all

$$
e_i.
$$

Look for:

- concentration near zero;
- long tails;
- multiple modes;
- isolated large errors.

### 12.4 Detected Chessboard Corners

This verifies that the observation data are correct **before** interpreting the calibration.

If corner ordering or detection is wrong, everything after it becomes meaningless.

### 12.5 Estimated Camera Poses

The plotted camera centre is

$$
C=-R^Tt.
$$

This figure should show plausible relative camera locations around the chessboard.

### 12.6 Detected vs Reprojected Points

This is one of the strongest qualitative checks.

Measured corners and predicted corners should nearly overlap.

Systematic separation patterns are informative:

- radial pattern → possible lens distortion;
- global shift → possible translation/principal-point issue;
- directional stretch → possible focal-scale issue;
- one bad image → possible detection or pose problem.

### Technical Implication

A calibration result is strongest when:

$$
\text{small numerical error}
+
\text{good visual overlap}
+
\text{physically plausible poses}.
$$

## 13. Run the Numerical and Output-file Validation Checks

### Validation Rationale

A notebook can execute without crashing and still produce invalid geometry.

The final checks turn mathematical assumptions into explicit tests.

### Check 1 — Intrinsic matrix shape and finiteness

The code verifies

$$
K\in\mathbb{R}^{3\times3}
$$

and that every entry is finite.

NaN or infinity usually indicates degeneracy or invalid square-root/division operations.

### Check 2 — Rotation orthonormality

A true rotation matrix satisfies

$$
R^TR=I.
$$

The implementation checks this numerically with a tolerance.

### Check 3 — Proper rotation determinant

A valid 3D rotation satisfies

$$
\det(R)=+1.
$$

A determinant near $-1$ would indicate a reflection.

### Check 4 — Finite reprojection errors

All $e_i$ must be finite.

Non-finite errors imply failed projection geometry, usually because of invalid $K$, $R$, $t$, or division by an invalid homogeneous coordinate.

### Check 5 — Required figures exist

The implementation confirms that all six diagnostic PNG files were actually created.

This validates not only mathematics but also reproducibility of the project output.

### Physical and Numerical Plausibility

Beyond the automated tests, an expert should also ask:

- Are $\alpha$ and $\beta$ plausible for the image resolution?
- Is $(u_0,v_0)$ reasonably located?
- Is $\gamma$ small?
- Are camera poses physically plausible?
- Are some views much worse than others?
- Do residuals show systematic spatial structure?


## Technical Synthesis

The calibration model is fully determined by the following dependency chain:

$$
\boxed{
\text{Images}
\rightarrow
\mathbf{x}_i
\rightarrow
\mathbf{X}_{p,i}
\rightarrow
T
\rightarrow
Q
\rightarrow
H
\rightarrow
V
\rightarrow
b
\rightarrow
K
\rightarrow
R,t
\rightarrow
\hat{\mathbf{x}}_i
\rightarrow
e_i
}
$$

Each element has a direct implementation counterpart: detected image points, planar coordinates, normalization transforms, DLT systems, Zhang constraints, intrinsic recovery, pose estimation, reprojection, and residual analysis. The credibility of the calibration therefore rests on geometric consistency, numerical conditioning, physically valid rotations, and the spatial structure of the reprojection residuals.

## Scope and Limitations

The implemented model is deliberately focused.

### Included

- planar chessboard calibration;
- normalized DLT;
- Zhang closed-form intrinsic calibration;
- pose recovery;
- SVD-based rotation correction;
- reprojection analysis.

### Not included

- radial distortion coefficients;
- tangential distortion coefficients;
- nonlinear bundle adjustment;
- uncertainty propagation;
- robust outlier rejection;
- automatic model selection.

Because distortion is not modeled, systematic residual patterns near image borders may remain even when the linear calibration is otherwise correct.

That limitation is not a contradiction: it defines the exact scope of the solved laboratory problem.